In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial.distance import pdist, squareform
import networkx as nx

## Defining Dataset

In [ ]:
df_summary = pd.read_csv('../../References/London_Multiplex_Transport/Dataset/london_transport_disruptions_summary.txt', sep = ' ', engine='python')
df_summary.head(10)

In [ ]:
nodes_df = pd.read_csv('../../References/London_Multiplex_Transport/Dataset/london_transport_nodes.txt', sep = ' ', engine='python')
nodes_df.head(10)

In [ ]:
layers = pd.read_csv('../../References/London_Multiplex_Transport/Dataset/london_transport_layers.txt', sep = ' ', engine='python')
layers.head(10)

In [ ]:
edges_file = '../../References/London_Multiplex_Transport/Dataset/london_transport_raw.edges'
edges_df = pd.read_csv(edges_file, sep=' ', names=['Line', 'From Station', 'To Station'])

## Creating functions for the cartesian conversion, assigning line_color and line_stations

In [ ]:
def apply_repulsion(cartesian_df, iterations=100, repulsion_strength=0.01):
    """
    Apply a simple repulsion force to spread out nodes to avoid overlapping.
    
    Parameters:
    - cartesian_df (pd.DataFrame): DataFrame with Cartesian coordinates.
    - iterations (int): Number of iterations for the force-directed algorithm.
    - repulsion_strength (float): Strength of the repulsion force.
    
    Returns:
    - pd.DataFrame: Updated DataFrame with adjusted Cartesian coordinates.
    """
    coords = cartesian_df[['x', 'y']].values
    for _ in range(iterations):
        distances = squareform(pdist(coords))
        # Avoid division by zero and ensure valid distances
        np.fill_diagonal(distances, np.inf)
        with np.errstate(divide='ignore', invalid='ignore'):
            repulsion = np.nan_to_num(repulsion_strength / distances**2, nan=0.0, posinf=0.0, neginf=0.0)
        force_x = np.sum(repulsion * (coords[:, 0][:, np.newaxis] - coords[:, 0]), axis=1)
        force_y = np.sum(repulsion * (coords[:, 1][:, np.newaxis] - coords[:, 1]), axis=1)
        coords[:, 0] += force_x
        coords[:, 1] += force_y
    
    cartesian_df['x'] = coords[:, 0]
    cartesian_df['y'] = coords[:, 1]
    return cartesian_df

def convert_to_cartesian(nodes_df, figsize=(45, 20), iterations=100, repulsion_strength=0.01):
    """
    Convert latitude and longitude to Cartesian coordinates for plotting.
    
    Parameters:
    - nodes_df (pd.DataFrame): DataFrame with columns ['nodeID', 'nodeLabel', 'nodeLat', 'nodeLong'].
    - figsize (tuple): Size of the figure for plotting.
    - iterations (int): Number of iterations for the force-directed algorithm.
    - repulsion_strength (float): Strength of the repulsion force.
    
    Returns:
    - pd.DataFrame: DataFrame with columns ['nodeID', 'nodeLabel', 'x', 'y'] for Cartesian coordinates.
    """
    # Ensure there are no NaN values in latitude and longitude
    nodes_df = nodes_df.dropna(subset=['nodeLat', 'nodeLong'])
    
    # Extract latitudes and longitudes
    latitudes = nodes_df['nodeLat'].values
    longitudes = nodes_df['nodeLong'].values
    
    # Normalize latitudes and longitudes to range [0, 1]
    scaler = MinMaxScaler()
    normalized_coords = scaler.fit_transform(np.column_stack((latitudes, longitudes)))
    
    # Scale to the size of the figure
    fig_width, fig_height = figsize
    scaled_lats = normalized_coords[:, 0] * fig_height
    scaled_longs = normalized_coords[:, 1] * fig_width
    
    # Create a DataFrame with Cartesian coordinates
    cartesian_df = pd.DataFrame({
        'nodeID': nodes_df['nodeID'],
        'nodeLabel': nodes_df['nodeLabel'],
        'x': scaled_longs,
        'y': scaled_lats
    })
    
    # Apply repulsion to spread out nodes
    cartesian_df = apply_repulsion(cartesian_df, iterations, repulsion_strength)
    
    return cartesian_df

cartesian_df = convert_to_cartesian(nodes_df)

print(cartesian_df)

cartesian_df.to_csv('cartesian.csv', sep=',', encoding='utf-8', index=False, header=True)


In [ ]:
def assign_line_colors(edges_df):
    line_colors = {
        'bakerloo': 'brown',
        'circle': 'yellow',
        'district': 'green',
        'hammersmith&city': 'pink',
        'jubilee': 'gray',
        'metropolitan': 'magenta',
        'northern': 'black',
        'piccadilly': 'blue',
        'victoria': 'lightblue',
        'waterloo&city': 'cyan',
        'overground': 'orange',
        'dlr': 'turquoise',
        'central': 'red'  # Adding Central line for completeness
        
    }
    
    line_stations = {}
    for line in line_colors:
        line_stations[line] = set(edges_df[edges_df['Line'] == line]['From Station']).union(set(edges_df[edges_df['Line'] == line]['To Station']))
    
    return line_colors, line_stations

## Plotting the network of the whole transit

In [ ]:
# Converting Latitude and Longitude coordinates of the df to cartesian coordinates
cartesian_df = convert_to_cartesian(nodes_df)

# Assign colors to lines
line_colors, line_stations = assign_line_colors(edges_df)

In [ ]:
# Create a graph
G = nx.Graph()

# Add nodes with positions
pos = {}
for _, row in cartesian_df.iterrows():
    pos[row['nodeLabel']] = (row['x'], row['y'])

# Add nodes with colors
node_color_map = {}
for line, stations in line_stations.items():
    for station in stations:
        G.add_node(station)
        if station in pos:
            node_color_map[station] = line_colors[line]
        else:
            node_color_map[station] = 'gray'  # Station not in nodes_df

# Add edges with colors
for _, row in edges_df.iterrows():
    line = row['Line']
    if row['From Station'] in line_stations[line] and row['To Station'] in line_stations[line]:
        G.add_edge(row['From Station'], row['To Station'], color=line_colors[line])
    else:
        G.add_edge(row['From Station'], row['To Station'], color='gray')  # Edge not valid

# Draw the graph
plt.figure(figsize=(35, 20))
nx.draw(G, pos, with_labels=True, node_size=500, font_size=12, node_color=list(node_color_map.values()), font_color='black', edge_color=[G[u][v]['color'] for u, v in G.edges()], width=2, alpha=0.7)

# Create custom legend for line colors
legend_patches = []
for line, color in line_colors.items():
    legend_patches.append(plt.Line2D([], [], color=color, label=line.capitalize(), linestyle='-'))

plt.legend(handles=legend_patches, fontsize=20)

plt.title('TfL Station Network - 2014', fontsize=40)
plt.show()

## Calculating each Centrality Measure

In [ ]:
def plot_degree_centrality(G):

    cartesian_df = convert_to_cartesian(nodes_df)
    line_colors, line_stations = assign_line_colors(edges_df)

    plt.figure(figsize=(45, 20))
    
    # Calculate degree centrality
    degree_centrality = nx.degree_centrality(G)
    top_5_degree = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
    top_5_nodes = [node for node, _ in top_5_degree]

    # Prepare node sizes and colors
    node_sizes = [1500 if node in top_5_nodes else 500 for node in G.nodes()]
    node_colors = []
    for node in G.nodes():
        for line, stations in line_stations.items():
            if node in stations:
                node_colors.append(line_colors[line])
                break
        else:
            node_colors.append('gray')

    # Prepare node positions
    pos = {row['nodeLabel']: (row['x'], row['y']) for _, row in cartesian_df.iterrows()}

    # Draw the graph
    nx.draw(G, pos, node_size=node_sizes, node_color=node_colors, 
            edge_color=[G[u][v]['color'] for u, v in G.edges()], width=2, alpha=0.7)

    # Add labels
    for node, (x, y) in pos.items():
        if node in top_5_nodes:
            plt.text(x, y, node, fontsize=14, fontweight='bold', ha='center', va='center')
        else:
            plt.text(x, y, node, fontsize=10, ha='center', va='center')

    # Create legend
    legend_patches = [plt.Line2D([], [], color=color, label=line.capitalize(), linestyle='-') 
                      for line, color in line_colors.items()]
    plt.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1, 1))

    plt.title('TfL Station Network - Degree Centrality', fontsize=20)
    plt.show()

In [ ]:
plot_degree_centrality(G)

In [ ]:
def plot_betweenness_centrality(G):


    cartesian_df = convert_to_cartesian(nodes_df)
    line_colors, line_stations = assign_line_colors(edges_df)

    plt.figure(figsize=(45, 20))
    
    # Calculate betweenness centrality
    betweenness_centrality = nx.betweenness_centrality(G)
    top_5_betweenness = sorted(betweenness_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
    top_5_nodes = [node for node, _ in top_5_betweenness]

    # Prepare node sizes and colors
    node_sizes = [1500 if node in top_5_nodes else 500 for node in G.nodes()]
    node_colors = []
    for node in G.nodes():
        for line, stations in line_stations.items():
            if node in stations:
                node_colors.append(line_colors[line])
                break
        else:
            node_colors.append('gray')

    # Prepare node positions
    pos = {row['nodeLabel']: (row['x'], row['y']) for _, row in cartesian_df.iterrows()}

    # Draw the graph
    nx.draw(G, pos, node_size=node_sizes, node_color=node_colors, 
            edge_color=[G[u][v]['color'] for u, v in G.edges()], width=2, alpha=0.7)

    # Add labels
    for node, (x, y) in pos.items():
        if node in top_5_nodes:
            plt.text(x, y, node, fontsize=14, fontweight='bold', ha='center', va='center')
        else:
            plt.text(x, y, node, fontsize=10, ha='center', va='center')

    # Create legend
    legend_patches = [plt.Line2D([], [], color=color, label=line.capitalize(), linestyle='-') 
                      for line, color in line_colors.items()]
    plt.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1, 1))

    plt.title('TfL Station Network - Betweenness Centrality', fontsize=20)
    plt.show()

plot_betweenness_centrality(G)

In [ ]:
def plot_closeness_centrality(G):

    cartesian_df = convert_to_cartesian(nodes_df)
    line_colors, line_stations = assign_line_colors(edges_df)
    
    plt.figure(figsize=(45, 20))
    
    # Calculate closeness centrality
    closeness_centrality = nx.closeness_centrality(G)
    top_5_closeness = sorted(closeness_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
    top_5_nodes = [node for node, _ in top_5_closeness]

    # Prepare node sizes and colors
    node_sizes = [1500 if node in top_5_nodes else 500 for node in G.nodes()]
    node_colors = []
    for node in G.nodes():
        for line, stations in line_stations.items():
            if node in stations:
                node_colors.append(line_colors[line])
                break
        else:
            node_colors.append('gray')

    # Prepare node positions
    pos = {row['nodeLabel']: (row['x'], row['y']) for _, row in cartesian_df.iterrows()}

    # Draw the graph
    nx.draw(G, pos, node_size=node_sizes, node_color=node_colors, 
            edge_color=[G[u][v]['color'] for u, v in G.edges()], width=2, alpha=0.7)

    # Add labels
    for node, (x, y) in pos.items():
        if node in top_5_nodes:
            plt.text(x, y, node, fontsize=14, fontweight='bold', ha='center', va='center')
        else:
            plt.text(x, y, node, fontsize=10, ha='center', va='center')

    # Create legend
    legend_patches = [plt.Line2D([], [], color=color, label=line.capitalize(), linestyle='-') 
                      for line, color in line_colors.items()]
    plt.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1, 1))

    plt.title('TfL Station Network - Closeness Centrality', fontsize=20)
    plt.show()

plot_closeness_centrality(G)

In [ ]:
def plot_eigenvector_centrality(G):

    cartesian_df = convert_to_cartesian(nodes_df)
    line_colors, line_stations = assign_line_colors(edges_df)

    plt.figure(figsize=(45, 20))
    
    # Calculate eigenvector centrality
    eigenvector_centrality = nx.eigenvector_centrality(G)
    top_5_eigenvector = sorted(eigenvector_centrality.items(), key=lambda x: x[1], reverse=True)[:5]
    top_5_nodes = [node for node, _ in top_5_eigenvector]

    # Prepare node sizes and colors
    node_sizes = [1500 if node in top_5_nodes else 500 for node in G.nodes()]
    node_colors = []
    for node in G.nodes():
        for line, stations in line_stations.items():
            if node in stations:
                node_colors.append(line_colors[line])
                break
        else:
            node_colors.append('gray')

    # Prepare node positions
    pos = {row['nodeLabel']: (row['x'], row['y']) for _, row in cartesian_df.iterrows()}

    # Draw the graph
    nx.draw(G, pos, node_size=node_sizes, node_color=node_colors, 
            edge_color=[G[u][v]['color'] for u, v in G.edges()], width=2, alpha=0.7)

    # Add labels
    for node, (x, y) in pos.items():
        if node in top_5_nodes:
            plt.text(x, y, node, fontsize=14, fontweight='bold', ha='center', va='center')
        else:
            plt.text(x, y, node, fontsize=10, ha='center', va='center')

    # Create legend
    legend_patches = [plt.Line2D([], [], color=color, label=line.capitalize(), linestyle='-') 
                      for line, color in line_colors.items()]
    plt.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1, 1))

    plt.title('TfL Station Network - Eigenvector Centrality', fontsize=20)
    plt.show()

plot_eigenvector_centrality(G)

In [ ]:
def plot_pagerank(G):

    cartesian_df = convert_to_cartesian(nodes_df)
    line_colors, line_stations = assign_line_colors(edges_df)

    plt.figure(figsize=(45, 20))
    
    # Calculate PageRank
    pagerank = nx.pagerank(G)
    top_5_pagerank = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]
    top_5_nodes = [node for node, _ in top_5_pagerank]

    # Prepare node sizes and colors
    node_sizes = [1500 if node in top_5_nodes else 500 for node in G.nodes()]
    node_colors = []
    for node in G.nodes():
        for line, stations in line_stations.items():
            if node in stations:
                node_colors.append(line_colors[line])
                break
        else:
            node_colors.append('gray')

    # Prepare node positions
    pos = {row['nodeLabel']: (row['x'], row['y']) for _, row in cartesian_df.iterrows()}

    # Draw the graph
    nx.draw(G, pos, node_size=node_sizes, node_color=node_colors, 
            edge_color=[G[u][v]['color'] for u, v in G.edges()], width=2, alpha=0.7)

    # Add labels
    for node, (x, y) in pos.items():
        if node in top_5_nodes:
            plt.text(x, y, node, fontsize=14, fontweight='bold', ha='center', va='center')
        else:
            plt.text(x, y, node, fontsize=10, ha='center', va='center')

    # Create legend
    legend_patches = [plt.Line2D([], [], color=color, label=line.capitalize(), linestyle='-') 
                      for line, color in line_colors.items()]
    plt.legend(handles=legend_patches, loc='upper left', bbox_to_anchor=(1, 1))

    plt.title('TfL Station Network - PageRank', fontsize=20)
    plt.show()

plot_pagerank(G)

### Summarizing the calculated centrality measure

In [ ]:
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)
closeness_centrality = nx.closeness_centrality(G)
eigenvector_centrality = nx.eigenvector_centrality(G)
pagerank = nx.pagerank(G)

# Function to get top N nodes for a centrality measure
def get_top_nodes(centrality_dict, n=5):
    return sorted(centrality_dict.items(), key=lambda x: x[1], reverse=True)[:n]

# Print top 5 nodes for each centrality measure
print("Top 5 nodes by Degree Centrality: \n", get_top_nodes(degree_centrality), "\n")
print("Top 5 nodes by Betweenness Centrality: \n", get_top_nodes(betweenness_centrality), "\n")
print("Top 5 nodes by Closeness Centrality: \n", get_top_nodes(closeness_centrality), "\n")
print("Top 5 nodes by Eigenvector Centrality: \n", get_top_nodes(eigenvector_centrality), "\n")
print("Top 5 nodes by PageRank: \n", get_top_nodes(pagerank), "\n")

In [ ]:
#wrote a function to get top N nodes for a centrality measure
def get_top_nodes(centrality_dict, n=5):
    return [node for node, _ in sorted(centrality_dict.items(), key=lambda x: x[1], reverse=True)[:n]]

In [ ]:
top_degree = get_top_nodes(degree_centrality)
top_betweenness = get_top_nodes(betweenness_centrality)
top_closeness = get_top_nodes(closeness_centrality)
top_eigenvector = get_top_nodes(eigenvector_centrality)
top_pagerank = get_top_nodes(pagerank)

# Combine all top nodes into a single list
all_top_nodes = top_degree + top_betweenness + top_closeness + top_eigenvector + top_pagerank

# Count the occurrences of each node
node_counts = pd.Series(all_top_nodes).value_counts()

# Identify nodes that appear most frequently across all centrality measures
top_combined_nodes = node_counts[node_counts > 1].index.tolist()

# Print the top combined nodes
print("Nodes with the highest centrality across multiple measures: \n", top_combined_nodes)

## Plotting with Cooordinates

In [ ]:
G = nx.Graph()

# Add nodes with positions
pos = {}
for _, row in nodes_df.iterrows():
    pos[row['nodeLabel']] = (row['nodeLong'], row['nodeLat'])

# Add nodes with colors
node_color_map = {}
for line, stations in line_stations.items():
    for station in stations:
        G.add_node(station)
        if station in pos:
            node_color_map[station] = line_colors[line]
        else:
            node_color_map[station] = 'gray'  # Station not in nodes_df

# Add edges with colors
for _, row in edges_df.iterrows():
    line = row['Line']
    if row['From Station'] in line_stations[line] and row['To Station'] in line_stations[line]:
        G.add_edge(row['From Station'], row['To Station'], color=line_colors[line])
    else:
        G.add_edge(row['From Station'], row['To Station'], color='gray')  # Edge not valid

# Draw the graph
plt.figure(figsize=(55, 40))
nx.draw(G, pos, with_labels=True, node_size=500, font_size=8, node_color=list(node_color_map.values()), font_color='black', edge_color=[G[u][v]['color'] for u, v in G.edges()], width=3, alpha=0.7)

# Create custom legend for line colors
legend_patches = []
for line, color in line_colors.items():
    legend_patches.append(plt.Line2D([], [], color=color, label=line.capitalize(), linestyle='-'))

plt.legend(handles=legend_patches)
plt.xlim(-0.63, 0.27)
plt.ylim(51.37, 51.71)
plt.title('TfL Station Network with Multiple Tube Lines Highlighted', fontsize=30)
plt.show()